In [2]:
import numpy as np
from numba import njit, prange
import matplotlib.pyplot as plt
from tqdm import tqdm
from glob import glob
import os
%matplotlib tk

In [3]:
PIXMAP = np.array([190,230,138,62,254,274,172,96,20,310,220,130,54,182,264,164,88,12,302,208,
                   122,46,299,252,156,80,4,294,196,114,38,262,242,148,72,263,286,186,106,30,174,
                   232,140,64,270,276,176,98,22,312,222,132,56,210,266,166,90,14,304,212,124,48,
                   255,256,158,82,6,296,200,116,40,278,244,150,74,291,288,188,108,32,218,236,142,
                   66,279,280,178,100,24,314,224,134,58,226,268,168,92,16,306,214,126,50,283,258,
                   160,84,8,298,204,118,42,271,248,152,76,0,290,192,110,34,202,238,144,68,235,282,
                   180,102,26,316,228,136,60,234,272,170,94,18,308,216,128,52,198,260,162,86,10,
                   300,206,120,44,247,250,154,78,2,292,194,112,36,246,240,146,70,307,284,184,104,
                   28,318,267,59,141,223,7,89,171,269,37,119,201,315,67,149,231,15,97,179,285,45,
                   127,209,295,75,157,241,23,105,187,301,53,135,217,1,83,165,257,31,113,195,243,
                   61,143,225,9,91,173,273,39,121,203,311,69,151,233,17,99,181,289,47,129,211,287,
                   77,159,245,25,107,189,305,55,137,219,3,85,167,261,33,115,197,303,63,145,227,11,
                   93,175,277,41,123,205,259,71,153,237,19,101,183,293,49,131,213,319,79,161,249,
                   27,109,191,309,57,139,221,5,87,169,265,35,117,199,275,65,147,229,13,95,177,281,
                   43,125,207,251,73,155,239,21,103,185,297,51,133,215,317,81,163,253,29,111,193,313])

def get_pixID(pix):
    """Sensor location -> (pixel_ID, is_master)"""
    pix_idx = np.argwhere(PIXMAP == pix)[0][0]
    is_master = int(pix_idx >= 170)
    pix_ID = pix_idx - 170 * is_master
    return pix_ID, is_master

def get_pixLoc(pix_ID, is_master):
    """(pixel_ID, is_master) -> sensor location"""
    return PIXMAP[pix_ID + 170 * is_master]

In [4]:
def concat_picoseconds(directory):
    """Convert raw .txt files into one binary file per pixel and per sync signal.

    Physical pixels  -> px_000.bin ... px_319.bin  (named by sensor location)
    Sync signals     -> master_dwell.bin / slave_dwell.bin  (pixel ID 225)
                        master_line.bin  / slave_line.bin   (pixel ID 226)
                        master_frame.bin / slave_frame.bin  (pixel ID 228)

    Each file is a flat array of int64 absolute timestamps in picoseconds,
    written in chronological order (already sorted).
    """
    import pandas as pd

    SPECIAL          = {225: 'dwell', 226: 'line', 228: 'frame'}
    VALID_SPECIAL    = np.array(list(SPECIAL.keys()), dtype=np.uint16)
    sec_per_count    = 1 / 10e6
    ps_per_count     = int(sec_per_count * 1e12)
    counts_per_reset = 2**16
    chunksize        = 1_000_000

    # Precompute raw pixel_id -> sensor location for each chip
    master_loc = np.array([get_pixLoc(i, 1) for i in range(150)])  # master: IDs 0-149
    slave_loc  = np.array([get_pixLoc(i, 0) for i in range(170)])  # slave:  IDs 0-169

    files        = sorted(glob(os.path.join(directory, '*.txt')))
    master_files = [f for f in files if 'master' in os.path.basename(f)]
    slave_files  = [f for f in files if 'slave'  in os.path.basename(f)]

    # Open all output files upfront
    handles = {}
    for loc in range(320):
        handles[loc] = open(os.path.join(directory, f'px_{loc:03d}.bin'), 'wb')
    for chip in ('master', 'slave'):
        for name in SPECIAL.values():
            handles[(chip, name)] = open(os.path.join(directory, f'{chip}_{name}.bin'), 'wb')

    def process_chip(chip_files, loc_map, chip_name):
        n_phys = len(loc_map)  # 150 for master, 170 for slave
        reset_count = 0
        for file in tqdm(chip_files, desc=chip_name):
            reader = pd.read_csv(
                file, sep=',', header=None,
                names=['pixel_id', 'counter', 'timestamp'],
                dtype={'pixel_id': np.uint16, 'counter': np.uint16, 'timestamp': np.float32},
                chunksize=chunksize, engine='c',
            )
            for chunk in reader:
                pid = chunk['pixel_id'].to_numpy()
                ctr = chunk['counter'].to_numpy()
                ts  = chunk['timestamp'].to_numpy()

                reset_count += int(np.sum(pid == 234))

                valid = (pid < n_phys) | np.isin(pid, VALID_SPECIAL)
                pid = pid[valid]
                ctr = ctr[valid]
                ts  = np.nan_to_num(ts[valid], nan=0.0).astype(np.int64)

                time_ps = (
                    (reset_count * counts_per_reset + ctr.astype(np.int64))
                    * ps_per_count + ts
                )

                # Physical pixels: group by raw ID, write to sensor-location file
                phys_mask = pid < n_phys
                if phys_mask.any():
                    phys_pid = pid[phys_mask]
                    phys_ts  = time_ps[phys_mask]
                    for uid in np.unique(phys_pid):
                        mask = phys_pid == uid
                        handles[loc_map[uid]].write(phys_ts[mask].tobytes())

                # Sync signals
                for sp_id, name in SPECIAL.items():
                    mask = pid == sp_id
                    if mask.any():
                        handles[(chip_name, name)].write(time_ps[mask].tobytes())

    try:
        process_chip(master_files, master_loc, 'master')
        process_chip(slave_files,  slave_loc,  'slave')
    finally:
        for h in handles.values():
            h.close()

    print('Done.')

In [5]:
@njit(parallel=True)
def _multistart_multistop_numba(t1, t2, idx, bin_width, tmax, nbins, n_shift):
    hist_private = np.zeros((2 * n_shift, nbins), dtype=np.int64)

    for s in prange(-n_shift, n_shift):
        s_idx = s + n_shift
        for i in range(len(t1)):
            j = idx[i] + s
            if 0 <= j < len(t2):
                tau = t2[j] - t1[i]
                b = int(np.floor((tau + tmax) / bin_width))
                if 0 <= b < nbins:
                    hist_private[s_idx, b] += 1

    return hist_private.sum(axis=0)


def create_multistart_multistop(path1, path2, bin_width, tmax, n_shift=20):
    """Compute g2 histogram between two pixel timestamp files.

    path1, path2  : paths to px_*.bin files (int64 picosecond timestamps, sorted)
    bin_width, tmax: in picoseconds
    Returns (hist, bins). The files are memory-mapped; no full array is loaded into RAM.
    """
    t1 = np.memmap(path1, dtype=np.int64, mode='r')
    t2 = np.memmap(path2, dtype=np.int64, mode='r')

    bins  = np.arange(-tmax - bin_width / 2, tmax + 3 * bin_width / 2, bin_width)
    nbins = len(bins) - 1

    # idx is the only array allocated in RAM (same length as t1, dtype int64)
    idx = np.searchsorted(t2, t1)

    hist = _multistart_multistop_numba(t1, t2, idx, bin_width, tmax, nbins, n_shift)
    return hist, bins

In [6]:
def create_multistart_multistop_chunked(path1, path2, bin_width, tmax, n_shift=20, chunk=1_000_000, offset=0):
    """Chunked g2 histogram between two pixel timestamp files.

    path1, path2   : paths to px_*.bin files (int64 picoseconds, sorted)
    bin_width, tmax: in picoseconds
    offset         : constant time shift dt (in picoseconds) such that
                     t2_stored = t2_actual + dt.
                     The corrected delay is tau = (t2_stored[j] - offset) - t1[i].
                     Default 0 (no correction).

    Reads t1 in chunks and slides a matching window through t2.
    Only one t1 chunk + the corresponding t2 window are in RAM at any time.
    Returns (hist, bins).
    """
    t1 = np.memmap(path1, dtype=np.int64, mode='r')
    t2 = np.memmap(path2, dtype=np.int64, mode='r')

    bins  = np.arange(-tmax - bin_width / 2, tmax + 3 * bin_width / 2, bin_width)
    nbins = len(bins) - 1
    hist  = np.zeros(nbins, dtype=np.int64)

    # Skip t2 elements that are too early to ever contribute
    # t2_corrected[j] = t2_stored[j] - offset >= t1[0] - tmax
    # => t2_stored[j] >= t1[0] - tmax + offset
    j_lo = max(0, int(np.searchsorted(t2, t1[0] - tmax + offset)) - n_shift)

    for i_lo in tqdm(range(0, len(t1), chunk), desc='correlating'):
        t1_chunk = np.array(t1[i_lo : i_lo + chunk])  # load chunk into RAM

        # one binary search on the t2 memmap to find the upper window bound
        # t2_stored[j] <= t1_chunk[-1] + tmax + offset covers all contributing events
        j_hi = int(np.searchsorted(t2, t1_chunk[-1] + tmax + offset)) + n_shift + 1
        j_hi = min(j_hi, len(t2))

        t2_window    = np.array(t2[j_lo:j_hi])    # load window into RAM
        t2_corrected = t2_window - offset          # apply offset in RAM (no memmap copy)

        idx_chunk = np.searchsorted(t2_corrected, t1_chunk)  # entirely in RAM

        hist += _multistart_multistop_numba(
            t1_chunk, t2_corrected, idx_chunk,
            bin_width, tmax, nbins, n_shift
        )

        # advance j_lo: next chunk needs no t2 earlier than this chunk's minimum
        j_lo += max(0, int(idx_chunk[0]) - n_shift)

    return hist, bins

In [12]:
path1 = '.\\spad_data\\node1\\px_215.bin'
path2 = '.\\spad_data\\node2\\px_211.bin'

dwell_path1_master = '.\\spad_data\\node1\\master_dwell.bin'
dwell_path2_master = '.\\spad_data\\node2\\master_dwell.bin'
dwell_path1_slave  = '.\\spad_data\\node1\\slave_dwell.bin'
dwell_path2_slave  = '.\\spad_data\\node2\\slave_dwell.bin'

last_dwell1_master = np.memmap(dwell_path1_master, dtype=np.int64, mode='r')[-1]
last_dwell2_master = np.memmap(dwell_path2_master, dtype=np.int64, mode='r')[-1]
# last_dwell1_slave = np.memmap(dwell_path1_slave, dtype=np.int64, mode='r')[-1]
# last_dwell2_slave = np.memmap(dwell_path2_slave, dtype=np.int64, mode='r')[-1]

master_offset = last_dwell2_master - last_dwell1_master
# slave_offset  = last_dwell2_slave  - last_dwell1_slave

print(f'master offset: {master_offset}')

bin_width = 50_000  # 10 ns in ps
tmax = 5_000_000  # 5 ms in ps
hist, bins = create_multistart_multistop_chunked(
    path1, path2, bin_width, tmax, n_shift=20, chunk=1_000_000, offset=master_offset
)

plt.figure()
plt.bar(bins[:-1] / 1e6, hist, width=bin_width / 1e6, align='edge')
plt.xlabel('Delay (ms)')
plt.ylabel('Coincidence count')
plt.title('g2 histogram between pixel 215 (node1) and pixel 211 (   node2)')




master offset: 115380651


correlating: 100%|██████████| 9/9 [00:00<00:00, 10.04it/s]


Text(0.5, 1.0, 'g2 histogram between pixel 215 (node1) and pixel 211 (   node2)')

In [ ]:
for j in range(13,21):
    path1 = f'D:\\sii1_ldls{j}\\'
    path2 = f'D:\\sii2_ldls{j}\\'
    
    concat_picoseconds(path1) # convert raw .txt files into binary .bin files (one per pixel and sync signal)
    files1 = next(os.walk(path1))[2]
    txtfiles1 = [f for f in files1 if f.endswith('.txt')]
    for f in txtfiles1: # delete the original .txt files to save space
        os.remove(os.path.join(path1, f))

    concat_picoseconds(path2) # convert raw .txt files into binary .bin files (one per pixel and sync signal)
    files2 = next(os.walk(path2))[2]
    txtfiles2 = [f for f in files2 if f.endswith('.txt')]
    for f in txtfiles2: # delete the original .txt files to save space
        os.remove(os.path.join(path2, f))

In [ ]:
bin_width = 200 # in picoseconds
tmax = 500_000 # 500 ns 

bins_all = np.arange(-tmax - bin_width / 2, tmax + 3 * bin_width / 2, bin_width)
hist_all = np.zeros(len(bins_all) - 1, dtype=np.int64)


for j in range(1,13): # correlating pixel 24 on detector 1 with pixel 26 on detector 2 
    path1 = f'D:\\sii1_ldls{j}\\px_024.bin' # pixel location 24 
    path2 = f'D:\\sii2_ldls{j}\\px_026.bin' # pixel location 26 

    dwell_path1 = f'D:\\sii1_ldls{j}\\slave_dwell.bin' # slave dwell signal (pixel ID 225)
    dwell_path2 = f'D:\\sii2_ldls{j}\\slave_dwell.bin' # slave dwell signal (pixel ID 225)

    dwell1 = np.memmap(dwell_path1, dtype=np.int64, mode='r')
    dwell2 = np.memmap(dwell_path2, dtype=np.int64, mode='r')

    last_dwell1 = dwell1[-1] if len(dwell1) > 0 else 0
    last_dwell2 = dwell2[-1] if len(dwell2) > 0 else 0
    offset = last_dwell2 - last_dwell1

    hist, bins = create_multistart_multistop_chunked(path1, path2, bin_width, tmax, n_shift=20, offset=offset)
    hist_all += hist


correlating: 100%|██████████| 2099/2099 [10:29<00:00,  3.34it/s]
